# 28BYJ-48 步进电机演示（Jetson Xavier NX）

使用 **ULN2003 驱动板**，引脚采用 `GPIO.BOARD`（40Pin 物理编号）。由你提供的 `StepperMotor.py` 转换而来，可在 Jupyter 中逐格运行。

| ULN2003 | Jetson Xavier NX |
|---|---|
| IN1 | BOARD 13 |
| IN2 | BOARD 16 |
| IN3 | BOARD 18 |
| IN4 | BOARD 22 |
| VCC (+) | **外部 5V 电源正极**（适用于 5V 型电机） |
| GND (-) | 外部电源负极，并与开发板 GND 共地（例如 BOARD 34） |

**先断电接线。** 电机五线插头插 ULN2003 的电机插座；不要将电机绕组直接接 GPIO。确认电机铭牌是 5V 型，供电电压要与铭牌一致。测试前确认 13/16/18/22 已配置成 GPIO 输出。

> 只在 Jetson 上执行；普通 Windows Jupyter 没有 `Jetson.GPIO`。运行完毕，记得执行最后的释放 GPIO 单元。

## 1. 导入库、定义电机控制类

保留原来的八相半步序列及正反转算法。仅增加中断时自动停止输出，并让 `close()` **只释放本电机占用的四个引脚**，避免影响同时使用的 MG996R 舵机。

In [1]:
import time
import Jetson.GPIO as GPIO

class StepperMotor:
    """28BYJ-48 + ULN2003，采用 Xavier NX 的 BOARD 物理引脚编号。"""

    IN1 = 13
    IN2 = 16
    IN3 = 18
    IN4 = 22

    # 原 StepperMotor.py 的八相半步驱动序列
    SEQUENCE = [
        [1, 0, 0, 0],
        [1, 1, 0, 0],
        [0, 1, 0, 0],
        [0, 1, 1, 0],
        [0, 0, 1, 0],
        [0, 0, 1, 1],
        [0, 0, 0, 1],
        [1, 0, 0, 1],
    ]

    def __init__(self):
        GPIO.setwarnings(False)
        GPIO.setmode(GPIO.BOARD)
        self.pins = [self.IN1, self.IN2, self.IN3, self.IN4]
        for pin in self.pins:
            GPIO.setup(pin, GPIO.OUT)
            GPIO.output(pin, GPIO.LOW)

    def _output(self, pattern):
        for pin, value in zip(self.pins, pattern):
            GPIO.output(pin, value)

    def rotate(self, steps, direction=1, delay=0.002):
        """steps: 半步数；direction: 1 正转、-1 反转；delay: 每半步延时(秒)。"""
        if direction not in (1, -1):
            raise ValueError("direction must be 1 or -1")
        if not isinstance(steps, int) or steps < 0:
            raise ValueError("steps must be a nonnegative integer")
        if delay <= 0:
            raise ValueError("delay must be > 0")

        index = 0
        try:
            for _ in range(steps):
                self._output(self.SEQUENCE[index])
                index = (index + direction) % len(self.SEQUENCE)
                time.sleep(delay)
        finally:
            self.stop()

    def stop(self):
        """关闭四相输出。"""
        for pin in self.pins:
            GPIO.output(pin, GPIO.LOW)

    def close(self):
        """停止电机，并仅释放步进电机使用的 GPIO。"""
        self.stop()
        GPIO.cleanup(self.pins)

print("StepperMotor 类已加载")

StepperMotor 类已加载


WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


## 2. 初始化 ULN2003

先运行此单元。运行失败请先核对接线、引脚复用及 `Jetson.GPIO` 的安装/权限。

In [2]:
motor = StepperMotor()
print("GPIO 初始化完成：BOARD 13、16、18、22")

GPIO 初始化完成：BOARD 13、16、18、22


## 3. 正转约 1/8 圈

原程序使用 512 半步；28BYJ-48 常按约 4096 半步/圈估算，实际角度存在齿轮传动误差。

In [4]:
motor.rotate(512, direction=1, delay=0.002)
print("正转完成")

正转完成


## 4. 反转约 1/8 圈

In [5]:
motor.rotate(512, direction=-1, delay=0.002)
print("反转完成")

反转完成


## 5. 结束演示，释放 GPIO

运行后若想再转动，需重新执行第 2 步初始化。若前面的运行被手动中断，也请执行本单元。

In [ ]:
motor.close()
print("步进电机已停止，GPIO 已释放")